# Scalare (tema)

Apasa **Copy to Drive** (sus, in stanga) inainte sa scrii ceva aici, altfel tot ce faci dispare cand inchizi tab-ul.

Ruleaza prima celula de mai jos, o singura data, ca sa aduci utilitarele cursului.

In [ ]:
import os, urllib.request
if not os.path.exists('onia.py'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/alexbirs27/iLoveBeer/main/cursuri/colab/onia.py', 'onia.py')
from onia import *
descarca('date/elevi.csv', 'date/case.csv', 'anim_normalizare.py', 'anim_scalare.py')
from anim_normalizare import *
from anim_scalare import *

Exercitiile de acasa. Testele ascunse verifica si cazurile limita (liste constante, valori extreme, coloane cu `std = 0`). Functiile intorc **liste** daca nu scrie altfel.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import make_pipeline

case = pd.read_csv('date/case.csv')
elevi = pd.read_csv('date/elevi.csv').drop_duplicates()

vin = load_wine()                       # 178 vinuri, 13 masuratori chimice, 3 soiuri (clase)
Xv, yv = vin.data, vin.target
Xtr, Xte, ytr, yte = train_test_split(Xv, yv, test_size=0.3, random_state=0, stratify=yv)

### 1. Z-score cu `ddof`

`standardizeaza(v, ddof)`: standardizeaza lista `v`. Abaterea standard se calculeaza impartind suma patratelor la `n - ddof` (`ddof=0` populatie, `ddof=1` esantion). Daca abaterea e 0, intoarce zerouri.

In [ ]:
def standardizeaza(v, ddof):
    pass  # TODO

In [ ]:
def _ref_z(v, ddof):
    a = np.array(v, dtype=float)
    s = a.std(ddof=ddof)
    return [0.0] * len(v) if s == 0 else list((a - a.mean()) / s)

verifica('standardizeaza', standardizeaza,
         [(([2, 4, 4, 4, 5, 5, 7, 9], 0), [-1.5, -0.5, -0.5, -0.5, 0, 0, 1, 2]), (([1, 3], 1), [-(0.5 ** 0.5), 0.5 ** 0.5])],
         ascunse=[(([5, 5, 5], 0), [0.0, 0.0, 0.0]), (([7, 7], 1), [0.0, 0.0])],
         referinta=_ref_z, generator=lambda: (random.sample(range(-30, 30), random.randint(3, 7)), random.randint(0, 1)),
         indiciu='ddof spune cu cat scazi n la impartire; verifica si std == 0', puncte=2)

### 2. Scalare robusta

`robust_scaleaza(v)`: `(x - mediana) / (q3 - q1)`, cu cuartilele din `np.percentile` (interpolare liniara, cea implicita). Daca IQR este 0, imparte la 1 (asa face si `RobustScaler`).

In [ ]:
def robust_scaleaza(v):
    pass  # TODO

In [ ]:
def _ref_rob(v):
    return [float(x) for x in RobustScaler().fit_transform(np.array(v, dtype=float).reshape(-1, 1)).ravel()]

verifica('robust_scaleaza', robust_scaleaza,
         [(([12, 15, 18, 20, 25, 30, 500],), [-8 / 11, -5 / 11, -2 / 11, 0, 5 / 11, 10 / 11, 480 / 11])],
         ascunse=[(([5, 5, 5, 5],), [0.0, 0.0, 0.0, 0.0]), (([1, 1, 1, 1, 9],), [0.0, 0.0, 0.0, 0.0, 8.0])],
         referinta=_ref_rob, generator=lambda: ([random.randint(-40, 40) for _ in range(random.randint(4, 9))],),
         indiciu='np.percentile(a, [25, 50, 75]) da q1, mediana, q3 dintr-o singura chemare', puncte=2)

### 3. Outlieri dupa z-score

`outlieri_z(v, prag)`: intoarce lista **indicilor** valorilor pentru care `|z| > prag` (z-score cu abatere de populatie), in ordine crescatoare. Cu asta poti gasi valorile ciudate inainte sa scalezi.

In [ ]:
def outlieri_z(v, prag):
    pass  # TODO

In [ ]:
def _ref_out(v, prag):
    a = np.array(v, dtype=float)
    return [int(i) for i in np.where(np.abs((a - a.mean()) / a.std()) > prag)[0]]

verifica('outlieri_z', outlieri_z,
         [(([10] * 9 + [100], 2), [9]), (([1, 2, 3, 4, 5], 2), []), (([1, 2, 3, 4, 5], 1), [0, 4])],
         referinta=_ref_out, generator=lambda: ([random.randint(0, 10) for _ in range(12)] + [random.choice([-60, 80])], 2),
         indiciu='calculeaza z pentru fiecare valoare, apoi pastreaza indicii cu |z| > prag', puncte=2)

### 4. Cat de mult "fuge" informatia

`diferenta_fuga(train, test)`: calculeaza z-score-ul **primei valori din test** in doua feluri: cu media si abaterea (populatie) de pe `train`, si cu cele de pe `train + test` la un loc (varianta cu scurgere). Intoarce valoarea absoluta a diferentei dintre cele doua.

In [ ]:
def diferenta_fuga(train, test):
    pass  # TODO

In [ ]:
def _ref_fuga(tr, te):
    tot = tr + te
    return abs((te[0] - np.mean(tr)) / np.std(tr) - (te[0] - np.mean(tot)) / np.std(tot))

verifica('diferenta_fuga', diferenta_fuga,
         [(([0, 10], [20]), 3 - 10 / (200 / 3) ** 0.5), (([1, 2, 3], [2]), 0.0)],
         referinta=_ref_fuga, generator=lambda: (random.sample(range(0, 30), 6), random.sample(range(-40, 80), 3)),
         indiciu='calculeaza z de doua ori: o data cu statisticile din train, o data cu cele din train + test', puncte=2)

### 5. KNN cu si fara scalare

`acuratete_knn(scalat)`: antreneaza `KNeighborsClassifier(5)` pe `Xtr, ytr` (setul `wine` din prima celula) si intoarce acuratetea pe `Xte, yte`. Daca `scalat=True`, scaleaza intai cu `StandardScaler`, **cu statisticile din train**. Ar trebui sa vezi circa `0.72` si circa `0.96`.

In [ ]:
def acuratete_knn(scalat):
    pass  # TODO

In [ ]:
verifica('acuratete_knn', acuratete_knn, [((False,), 39 / 54), ((True,), 52 / 54)],
         indiciu='fit pe Xtr, transform pe Xtr si pe Xte', puncte=2)

### 6. Un pipeline pentru regresia logistica

`construieste_pipeline()`: intoarce un `Pipeline` cu doi pasi, `StandardScaler` si `LogisticRegression(max_iter=1000)`, in aceasta ordine. Verificarea il evalueaza cu validare incrucisata pe `wine`.

In [ ]:
def construieste_pipeline():
    pass  # TODO

In [ ]:
def _verif_pipeline():
    p = construieste_pipeline()
    nume = [type(s[1]).__name__ for s in p.steps]
    return nume, bool(cross_val_score(p, Xv, yv, cv=5).mean() > 0.95)

verifica('construieste_pipeline', _verif_pipeline, [((), (['StandardScaler', 'LogisticRegression'], True))],
         indiciu='make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))', puncte=2)

### 7. Program: z-score

Pe prima linie este `n`, pe a doua `n` numere. Afiseaza, **cate unul pe linie**, z-score-ul fiecaruia (abaterea standard de populatie), cu **3 zecimale**. Daca toate numerele sunt egale, afiseaza `0.000` pentru fiecare.

In [ ]:
def main():
    pass  # TODO

In [ ]:
verifica_program('program z-score', main,
                 [(['8', '2 4 4 4 5 5 7 9'], '-1.500\n-0.500\n-0.500\n-0.500\n0.000\n0.000\n1.000\n2.000'),
                  (['3', '7 7 7'], '0.000\n0.000\n0.000'),
                  (['2', '-4 4'], '-1.000\n1.000')],
                 ascunse=[(['3', '1 2 3'], '-1.225\n0.000\n1.225'), (['1', '5'], '0.000'),
                          (['4', '10 20 30 40'], '-1.342\n-0.447\n0.447\n1.342')], puncte=2)

### 8. Repara codul: scaler-ul invatat pe tot

Functia trebuie sa scaleze `train` si `test` cu statisticile din `train`. Pare corecta, dar cu `train = [2, 4, 4, 4, 5, 5, 7, 9]` si `test = [5, 9]` nu iese `[0, 2]` pentru test. Uita-te in animatie de unde vin media si abaterea, apoi repar-o.

In [ ]:
%%vezi
train = [2, 4]
test = [6, 100]
toate = train + test
m = sum(toate) / len(toate)
s = 0
for x in toate:
    s = s + (x - m) ** 2
sd = (s / len(toate)) ** 0.5
rez_test = [(x - m) / sd for x in test]

In [ ]:
def scaleaza(train, test):
    toate = train + test
    m = sum(toate) / len(toate)
    sd = (sum((x - m) ** 2 for x in toate) / len(toate)) ** 0.5
    return [(x - m) / sd for x in train], [(x - m) / sd for x in test]

In [ ]:
verifica('scaleaza', scaleaza,
         [(([2, 4, 4, 4, 5, 5, 7, 9], [5, 9]), ([-1.5, -0.5, -0.5, -0.5, 0, 0, 1, 2], [0, 2])), (([0, 10], [20]), ([-1, 1], [3]))],
         ascunse=[(([1, 3], [3, 5, 1]), ([-1, 1], [1, 3, -1]))],
         indiciu='media si abaterea trebuie sa vina numai din train', puncte=2)

### 9. Repara codul: `fit_transform` pe test

Aceasta functie foloseste `StandardScaler` si are aceeasi greseala, ascunsa intr-un singur rand. Cauta-l.

In [ ]:
def pregateste(Xtr, Xte):
    sc = StandardScaler()
    A = sc.fit_transform(Xtr)
    B = sc.fit_transform(Xte)
    return A, B

In [ ]:
verifica('pregateste', pregateste,
         [((np.array([[0.0], [10.0]]), np.array([[20.0]])), (np.array([[-1.0], [1.0]]), np.array([[3.0]])))],
         ascunse=[((np.array([[1.0, 5.0], [3.0, 9.0]]), np.array([[5.0, 5.0], [1.0, 1.0]])),
                   (np.array([[-1.0, -1.0], [1.0, 1.0]]), np.array([[3.0, -1.0], [-1.0, -3.0]])))],
         indiciu='pe test nu mai inveti nimic, doar aplici', puncte=1)

### 10. Provocare: propriul tau `StandardScaler`

Fara NumPy si fara `sklearn`, pe liste de liste (fiecare sublista e un rand, coloanele sunt caracteristicile).

- `fit(X)` intoarce dictionarul `{'medie': [...], 'std': [...]}` cu media si abaterea standard (populatie) **pe fiecare coloana**. Daca o coloana are `std == 0`, pune `1.0` in loc (ca `sklearn`).
- `transform(X, params)` intoarce lista de liste standardizata cu parametrii dati. Astfel `transform(test, fit(train))` foloseste statisticile din train.

In [ ]:
def fit(X):
    pass  # TODO


def transform(X, params):
    pass  # TODO

In [ ]:
verifica('fit', fit, [(([[1, 10], [3, 30], [5, 20]],), {'medie': [3.0, 20.0], 'std': [(8 / 3) ** 0.5, (200 / 3) ** 0.5]})],
         ascunse=[(([[4, 1], [4, 3]],), {'medie': [4.0, 2.0], 'std': [1.0, 1.0]})], puncte=1,
         indiciu='pentru fiecare coloana j: scoate coloana intr-o lista, apoi medie si abatere; std 0 devine 1.0')


def _fit_apoi_transform(train, test):
    return transform(test, fit(train))


def _ref_sk(train, test):
    return StandardScaler().fit(train).transform(test).tolist()

verifica('transform', _fit_apoi_transform,
         [(([[1, 10], [3, 30], [5, 20]], [[3, 20], [5, 10]]), _ref_sk([[1, 10], [3, 30], [5, 20]], [[3, 20], [5, 10]]))],
         ascunse=[(([[4, 1], [4, 3]], [[9, 5]]), [[5.0, 3.0]])],
         referinta=_ref_sk,
         generator=lambda: ([[random.randint(0, 9), random.randint(0, 99), 7] for _ in range(5)],
                            [[random.randint(-5, 15), random.randint(0, 99), random.randint(0, 9)] for _ in range(3)]),
         indiciu='transform(test, fit(train)): statisticile din train, aplicate pe test', puncte=2)
scor()